# 🏠 Foundry Hosted Agent + AGT Governance
**Agent Governance Toolkit — Citadel Lab Integration**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/agent-governance-toolkit/blob/main/agent-governance-python/notebooks/05_foundry_hosted_agent_with_citadel.ipynb)

In this notebook you will:
- Create a governed agent using AGT's `StatelessKernel`
- Route requests through a simulated Citadel APIM gateway
- See AGT policy enforcement block dangerous actions in real time
- Export governance events to Citadel's observability pipeline
- Inspect the full audit trail

### Architecture

```
Foundry Hosted Agent
        │
        ▼
  AGT StatelessKernel     ← Layer 2: Agent Control Plane
   (policy enforcement)
        │
        ▼
  Citadel APIM Gateway    ← Layer 1: Governance Hub
   (rate limiting, routing)
        │
        ▼
  Citadel Audit Exporter  ← Observability Pipeline
   (Event Hub + App Insights)
```

> **No API key required** — this demo runs fully offline using mock backends.
> To connect to live Azure services, set the environment variables in the optional cells.

## Step 1 — Install the toolkit

In [ ]:
!pip install agent-governance-toolkit[full] -q

## Step 2 — Set up the AGT Governance Kernel

In [ ]:
from agent_os.stateless import StatelessKernel, ExecutionContext

# Create the stateless kernel with default policies:
#   read_only  - blocks file_write, database_write, send_email
#   no_pii     - blocks SSN, credit card, password patterns
#   strict     - requires approval for sensitive actions
kernel = StatelessKernel()

print("Available policies:")
for name, rules in kernel.policies.items():
    print(f"  {name}: {rules}")

print("\n\u2705 AGT kernel ready")

## Step 3 — Simulate a Foundry Hosted Agent

We simulate a Foundry hosted agent that processes user requests.
In a real deployment, this would use `azure-ai-projects` and the Foundry Responses API.
The key insight: every agent action goes through AGT's `kernel.execute()` before reaching the model.

In [ ]:
class FoundryGovernedAgent:
    """A Foundry hosted agent with AGT governance.

    In production, this wraps a real Foundry agent:
        from azure.ai.projects import AIProjectClient
        client = AIProjectClient(endpoint=..., credential=...)
        agent = client.agents.create_agent(model="gpt-4o", ...)

    For this demo, we simulate LLM responses and focus on the
    governance layer that wraps every action.
    """

    def __init__(self, agent_id: str, policies: list[str], kernel: StatelessKernel):
        self.agent_id = agent_id
        self.policies = policies
        self.kernel = kernel
        self.context = ExecutionContext(
            agent_id=agent_id,
            policies=policies,
        )

    async def run(self, action: str, params: dict) -> dict:
        """Execute an action through AGT governance."""
        result = await self.kernel.execute(action, params, self.context)

        # Thread forward the updated context (stateless pattern)
        if result.updated_context:
            self.context = result.updated_context

        return {
            "success": result.success,
            "data": result.data,
            "error": result.error,
            "signal": result.signal,
        }


# Create the governed agent with read_only + no_pii policies
agent = FoundryGovernedAgent(
    agent_id="citadel-foundry-agent",
    policies=["read_only", "no_pii"],
    kernel=kernel,
)

print(f"Agent: {agent.agent_id}")
print(f"Policies: {agent.policies}")
print("\u2705 Governed agent ready")

## Step 4 — Route Through Citadel Gateway

Citadel's APIM gateway sits in front of the model endpoint, providing
rate limiting, subscription validation, and request routing.

We use the `MockGateway` for offline testing. In production, replace
with `CitadelGatewayConfig.from_env()` to connect to a real gateway.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'examples', 'citadel-governed-agent', 'src'))

from citadel_config import MockGateway, CitadelGatewayConfig

gateway = MockGateway(rate_limit=10)

# Simulate a gateway-routed LLM call
response = gateway.process_request(
    endpoint="/deployments/gpt-4o/chat/completions",
    payload={"messages": [{"role": "user", "content": "Hello"}]},
)

print(f"Gateway response: status={response['status']}")
print(f"APIM request ID: {response.get('apim_request_id')}")
print(f"Remaining quota: {response.get('remaining')}")
print("\n\u2705 Citadel gateway ready")

## Step 5 — Test Policy Enforcement

Now let's run several agent actions and see how AGT enforces policies.
Safe reads are allowed; dangerous writes and PII exposure are blocked.

In [ ]:
import asyncio

test_actions = [
    # (action, params, expected)
    ("database_query", {"query": "SELECT name FROM customers"}, "ALLOWED (read query)"),
    ("database_write", {"query": "DELETE FROM users"}, "BLOCKED (read_only policy)"),
    ("file_write", {"path": "/etc/passwd", "content": "x"}, "BLOCKED (read_only policy)"),
    ("send_email", {"to": "user@contoso.com", "body": "Hi"}, "BLOCKED (read_only policy)"),
    ("chat", {"message": "What is the weather?"}, "ALLOWED (safe action)"),
    ("chat", {"message": "My password is hunter2"}, "BLOCKED (no_pii policy)"),
    ("chat", {"message": "SSN: 123-45-6789"}, "BLOCKED (no_pii: social_security)"),
    ("respond", {"message": "Summarize the report"}, "ALLOWED (safe action)"),
]

audit_trail = []

print(f"{'Action':<20} {'Status':<12} Expected")
print("-" * 70)

for action, params, expected in test_actions:
    result = await agent.run(action, params)
    status = "\u2705 ALLOWED" if result["success"] else "\ud83d\udeab BLOCKED"
    print(f"{action:<20} {status:<12} {expected}")

    audit_trail.append({
        "action": action,
        "params": params,
        "success": result["success"],
        "error": result.get("error"),
        "signal": result.get("signal"),
    })

## Step 6 — Export Events to Citadel

The `CitadelAuditExporter` sends governance events to Azure Event Hub
and Application Insights for Citadel's unified observability dashboard.

Without credentials, the exporter operates in local-only mode (events
are buffered but not sent). Set environment variables to enable export.

In [ ]:
from agent_os.exporters.citadel_exporter import (
    CitadelAuditExporter,
    GovernanceEvent,
    GovernanceEventType,
    Decision,
    CorrelationContext,
)

# Create exporter (local-only mode without Azure credentials)
exporter = CitadelAuditExporter.from_env()

# Convert audit trail entries to Citadel governance events
for entry in audit_trail:
    event = GovernanceEvent(
        event_type=(
            GovernanceEventType.POLICY_DECISION
            if entry["success"]
            else GovernanceEventType.POLICY_VIOLATION
        ),
        agent_id=agent.agent_id,
        action=entry["action"],
        decision=Decision.ALLOW if entry["success"] else Decision.DENY,
        policy_name="read_only+no_pii",
        detail=entry.get("error") or "Action permitted",
        correlation=CorrelationContext(
            apim_request_id=f"citadel-lab-{hash(entry['action']) % 10000:04d}",
            agt_decision_id=f"agt-{hash(str(entry)) % 100000:05d}",
        ),
    )
    exporter.export_event(event)

print(f"Exported {len(audit_trail)} governance events to Citadel exporter")
print(f"  Buffered: {len(exporter._buffer)} events")
print(f"  Total exported: {exporter._total_exported}")
print("\n\u2705 Events queued for Citadel observability pipeline")
print("\nTo enable live export, set:")
print("  CITADEL_EVENTHUB_CONNECTION_STRING=<your-connection-string>")
print("  CITADEL_APPINSIGHTS_CONNECTION_STRING=<your-connection-string>")

## Step 7 — View the Audit Trail

The governance audit trail provides a complete record of every policy
decision, enabling compliance reporting and forensic analysis.

In [ ]:
print("\n\u2500\u2500 Governance Audit Trail \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
for i, entry in enumerate(audit_trail, 1):
    status = "\u2705 ALLOWED" if entry["success"] else "\ud83d\udeab BLOCKED"
    print(f"  [{i}] {entry['action']}")
    print(f"       Status: {status}")
    if entry.get("error"):
        print(f"       Reason: {entry['error']}")
    if entry.get("signal"):
        print(f"       Signal: {entry['signal']}")
    print()

blocked = sum(1 for e in audit_trail if not e["success"])
allowed = len(audit_trail) - blocked
print(f"Summary: {allowed} allowed, {blocked} blocked out of {len(audit_trail)} total")
print(f"\nAgent history length: {len(agent.context.history)}")

## Step 8 — (Optional) Connect to Live Foundry Agent

To use a real Azure AI Foundry hosted agent, replace the mock with:

```python
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

foundry_agent = client.agents.create_agent(
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    name="citadel-governed-agent",
    instructions="You are a helpful assistant.",
)

# Wrap every Foundry call through AGT governance:
async def governed_chat(user_message: str):
    result = await kernel.execute(
        action="chat",
        params={"message": user_message},
        context=ExecutionContext(
            agent_id=foundry_agent.id,
            policies=["read_only", "no_pii"],
        ),
    )
    if not result.success:
        return f"Blocked: {result.error}"

    # Only call Foundry if AGT allows it
    thread = client.agents.create_thread()
    client.agents.create_message(
        thread_id=thread.id, role="user", content=user_message
    )
    run = client.agents.create_and_process_run(
        thread_id=thread.id, agent_id=foundry_agent.id
    )
    messages = client.agents.list_messages(thread_id=thread.id)
    return messages.data[0].content[0].text.value
```

Required environment variables:
- `FOUNDRY_PROJECT_ENDPOINT`
- `AZURE_AI_MODEL_DEPLOYMENT_NAME`

## ✅ What You Learned

- How to wrap a Foundry hosted agent with AGT governance using `StatelessKernel`
- How `read_only` and `no_pii` policies block dangerous actions before they reach the model
- How the Citadel APIM gateway provides rate limiting and request routing
- How `CitadelAuditExporter` sends governance events to Event Hub and App Insights
- How to inspect the audit trail for compliance reporting

### Citadel Blueprint Integration

| Citadel Layer | Component | Role |
|---|---|---|
| Layer 1: Governance Hub | APIM Gateway | Rate limiting, routing, subscription validation |
| Layer 2: Agent Control Plane | **AGT StatelessKernel** | Policy enforcement, PII detection, action blocking |
| Observability | Citadel Audit Exporter | Event Hub + App Insights telemetry |

### Next Steps

- Try the [Policy Enforcement 101 notebook →](./01_policy_enforcement_101.ipynb) for deeper policy customization
- Try the [MCP Security Proxy notebook →](./02_mcp_security_proxy.ipynb) to govern tool server connections
- See the [Citadel integration guide](https://microsoft.github.io/agent-governance-toolkit/integrations/citadel/) for production deployment